# Champs de probabilité mensuels (step 4) – à faire une seule fois

Version Python de `Monthly probability fields2.R` (non modifié).

Pour chaque **mois calendaire**, on prend tous les produits binaires de ce mois sur 1984–2021 :
- **clair** = nombre d'années où le pixel vaut 0 ou 1 (pas nuage) ;
- **clair < 29 années** → NA (pas assez d'observations) ;
- **neige dans 100 % des années claires** → **1** ;
- **pas de neige dans 100 % des années claires** → **2** ;
- **sinon** → NA.

Le seuil est 100 % dans le script R, malgré le nom « 80 ». Il est réglable ici (`frac`).

**Entrée :** tes stacks binaires `raster_stack_predi_<saison>_<a/b>.tif` (sorties du modèle : 1 neige, 0 pas de neige, NA nuage). Les dates de chaque bande sont reprises des listes `raster_names_*` de `Model per saison1.R`.
- DJF est coupé dans le temps : `_a` = bandes 1–73, `_b` = bandes 73–105, la bande 73 est en double.
- MAM, JJA et SON sont coupés en moitiés ouest et est.
- `MAM_b.tif` est vide, on utilise `MAM_b1.tif`.

**Sorties :**
- `snow_model/proba_field_<mois>.tif` sur la grille NDSI (1, 2, 0 = NA), utilisés par `CP_Snow_gapfilling_steps3-7.ipynb` ;
- `snow_model/proba_fields_R_names/<Month>.nc`, `<Month>_a.nc` (ouest) et `<Month>_b.nc` (est) : mêmes noms et même format que `Monthly probability fields2.R` (variable `layer`, grille R 14234 × 6200), utilisables directement dans `Step4_Monthly proba fields.ipynb`.

In [ ]:
import os, re, time
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

r_script = '../original_R/Model_per_saison1.R'
stacks_dir = '../data/original_binary_stacks'   # from Zenodo
out_pattern = '../data/aux/proba_field_%02d.tif'
min_clear = 29          # minimum number of clear years (R: < 29 -> 5 -> NA)
frac = 1.0              # 1.0 = always snow / always no snow (as the R script)

# stacks: (file, first column, [band indices (1-based)], index of the first name)
W_R, H_R = 14234, 6200
half = W_R // 2
parts = {
    'DJF': [('raster_stack_predi_DJF_a.tif', 0, list(range(1, 73)), 0),     # bands 1-72 (73 = duplicate of DJF_b 1)
            ('raster_stack_predi_DJF_b.tif', 0, list(range(1, 34)), 72)],
    'MAM': [('raster_stack_predi_MAM_a3.tif', 0, list(range(1, 109)), 0),
            ('raster_stack_predi_MAM_b1.tif', half, list(range(1, 109)), 0)],
    'JJA': [('raster_stack_predi_JJA_a.tif', 0, list(range(1, 109)), 0),
            ('raster_stack_predi_JJA_b.tif', half, list(range(1, 109)), 0)],
    'SON': [('raster_stack_predi_SON_a3.tif', 0, list(range(1, 110)), 0),
            ('raster_stack_predi_SON_b.tif', half, list(range(1, 110)), 0)],
}

# dates of the bands, from the lists of the R script
src = open(r_script, encoding='latin-1').read()
names = {}
for s in parts:
    block = src[src.index('raster_names_%s <- c(' % s):]
    block = block[:block.index(')')]
    names[s] = [(int(y), int(m)) for y, m in re.findall(r"(\d{4})_(\d{2})_", block)]
    print(s, len(names[s]), 'months', names[s][:2], '...', names[s][-1])

## Comptage des années claires, avec et sans neige, par mois calendaire

In [ ]:
n1 = {m: np.zeros((H_R, W_R), 'uint16') for m in range(1, 13)}     # years with snow
n0 = {m: np.zeros((H_R, W_R), 'uint16') for m in range(1, 13)}     # years without snow
used = {m: [] for m in range(1, 13)}

t0 = time.time()
for s, lst in parts.items():
    for f, col0, bands, first in lst:
        with rasterio.open(os.path.join(stacks_dir, f)) as r:
            for k, b in enumerate(bands):
                y, m = names[s][first + k]
                a = r.read(b)
                cols = slice(col0, col0 + a.shape[1])
                n1[m][:, cols] += (a == 1)
                n0[m][:, cols] += (a == 0)
                if col0 == 0:
                    used[m].append(y)
        print('%s %s done (%.0f s)' % (s, f, time.time() - t0))

for m in range(1, 13):
    print('month %02d: %i products %s ... %s' % (m, len(used[m]), used[m][:3], used[m][-3:]))

## Champs 1 / 2 sur la grille NDSI

In [ ]:
T_R = rasterio.open(os.path.join(stacks_dir, 'raster_stack_predi_DJF_a.tif')).transform
T_N = from_origin(5.699957268017272, 47.85026882419969, 0.0003407435000010325, 0.00034046861818428666)
W_N, H_N = 14234, 6199
MASK = rasterio.open('../data/aux/mask_all_ndsi_grid.tif').read(1) == 1
n_ch = MASK.sum()

# also written as in "Monthly probability fields2.R": <Month>.nc and the halves <Month>_a.nc (west) / <Month>_b.nc (east),
# variable 'layer' (1 = snow, 2 = no snow, NaN), on the grid of the R products (14234 x 6200)
import xarray as xr
r_dir = '../outputs/proba_fields_R_names'
os.makedirs(r_dir, exist_ok=True)
month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June', 7: 'July',
               8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
lon_R = T_R.c + (np.arange(W_R) + 0.5) * T_R.a
lat_R = T_R.f + (np.arange(H_R) + 0.5) * T_R.e

def write_R_names(field, m):
    layer = np.where(field == 0, np.nan, field).astype('float32')
    for suffix, cols in [('', slice(0, W_R)), ('_a', slice(0, half)), ('_b', slice(half, W_R))]:
        ds = xr.Dataset({'layer': (('latitude', 'longitude'), layer[:, cols])},
                        coords={'latitude': lat_R, 'longitude': lon_R[cols]})
        ds['layer'].attrs['description'] = '1 = snow in 100 % of the clear years, 2 = no snow in 100 % of the clear years (>= 29 clear years)'
        ds.attrs['crs'] = 'EPSG:4326'
        ds.to_netcdf('%s/%s%s.nc' % (r_dir, month_names[m], suffix), encoding={'layer': {'zlib': True}})

pct = {}
for m in range(1, 13):
    clear = n0[m] + n1[m]
    ok = clear >= min_clear
    field = np.zeros((H_R, W_R), 'uint8')
    field[ok & (n1[m] >= frac * clear)] = 1
    field[ok & (n0[m] >= frac * clear)] = 2
    write_R_names(field, m)
    out = np.zeros((H_N, W_N), 'uint8')
    reproject(field, out, src_transform=T_R, src_crs='EPSG:4326', dst_transform=T_N, dst_crs='EPSG:4326',
              src_nodata=0, dst_nodata=0, resampling=Resampling.nearest)
    out[~MASK] = 0
    with rasterio.open(out_pattern % m, 'w', driver='GTiff', width=W_N, height=H_N, count=1, dtype='uint8',
                       crs='EPSG:4326', transform=T_N, nodata=0, compress='deflate') as dst:
        dst.write(out, 1)
    pct[m] = (100 * (out == 1).sum() / n_ch, 100 * (out == 2).sum() / n_ch)
    print('month %02d: snow (1) %.3f %% | no snow (2) %.3f %%' % (m, *pct[m]))

## Contrôle : comparaison avec les % notés dans `Monthly probability fields2.R`

In [ ]:
import pandas as pd
R_land = [1.827911e-04, 5.785092e-03, 3.804772e-02, 1.089507e+00, 2.723145e+00, 1.831320e+01, 3.186041e+01, 4.370062e+01, 4.993906e+01, 3.172566e+01, 6.598915e+00, 3.548840e-01]
R_snow = [0.2680903, 0.8121286, 10.0396143, 7.4514205, 2.8339613, 0.6216158, 0.3365012, 0.2492085, 0.2236622, 0.2211797, 0.3184542, 0.1151757]
order = [12, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]                  # Dec ... Nov as in the R script
pd.DataFrame({'month': order,
              'snow R (%)': R_snow, 'snow Python (%)': [round(pct[m][0], 4) for m in order],
              'no snow R (%)': R_land, 'no snow Python (%)': [round(pct[m][1], 4) for m in order]})

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(24, 12))
for a, m in zip(ax.flat, order):
    f = rasterio.open(out_pattern % m).read(1)[::6, ::6].astype(float); f[f == 0] = np.nan
    a.imshow(f, cmap=plt.matplotlib.colors.ListedColormap(['#3a7bd5', '#9bc36b']), vmin=1, vmax=2, interpolation='nearest')
    a.set_title('month %02d' % m); a.axis('off')
plt.tight_layout()